# Where should new childcare facilities go? A step-by-step guide

This notebook takes you through the analysis one section at a time, for **your own city** or a small **demo city**.

**How to use it:** run one section at a time. Click into a code cell and press **Shift + Enter**, or use the ▶ button. Fill in what appears, click its button, and wait for the green tick before you go on. Your answers are saved after each section, so you can stop and come back: run the cells at the top again, enter the same city name, and only re-run what you want to change.

**Do not use Run All in sections 1-12.** The sections wait for you, so Run All would rush past them.

| Sections | What happens |
|---|---|
| 1 | Choose a city |
| 2-11 | Give the data, one input at a time. Each is checked at once |
| 12 | Review your choices and create the settings |
| 13 | Check the data and build the model inputs |
| 14 | Enter your Gurobi licence |
| 15-16 | Choose the number of new facilities **P** (default 3), solve the three models, then see the results |

**The demo city** skips straight from section 1 to section 13.

**The three models**

- **Utilitarian:** minimises the total walking time of all children (aggregate efficiency).
- **Individual Rawlsian:** protects the worst-served location in the priority group.
- **District Rawlsian:** protects the priority district whose average child walks longest.

Demand is **estimated potential demand**, not observed enrolment.

## Set up this notebook

Run this cell first, every time you open the notebook.

In [ ]:
%matplotlib inline
# Optional: to skip section 1, set this before running. Leave as None to choose below.
CITY = None  # "demo", or the path of a finished settings file
LICENCE = None  # "installed", "file" or "wls"; None means section 14 asks
LICENCE_FILE = None  # path of gurobi.lic, if LICENCE = "file"

try:
    from childcare_access import city_steps as steps
    from childcare_access import guided_run as guide
except ImportError as problem:
    raise SystemExit(
        f"The project is not fully installed ({problem}). Follow README.md, then restart this notebook."
    ) from None

setup = steps.Setup()
print("Ready. Go on to section 1.")

## 1. Choose a city

Type your city's name. If you started this city before, your saved answers come back.

In [ ]:
steps.start(setup, CITY);

## 2. Districts

*Needed: a map file of the city's districts.* Run the cell, choose your file, click **Read this file**, check the map, then save.

In [ ]:
steps.districts(setup);

## 3. Walking network

*Needed: nothing, if you have internet.* The notebook can download the network from OpenStreetMap for you.

In [ ]:
steps.network(setup);

## 4. Existing childcare

*Needed: your city's official list of childcare facilities, as a map file.*

In [ ]:
steps.facilities(setup);

## 5. How to estimate where children live

*Needed: a decision.* **No census grid? That is fine:** choose *district totals and buildings*.

In [ ]:
steps.demand(setup);

## 6. Children per district

*Needed for the census-grid and district-total methods: a table.*

In [ ]:
steps.population(setup);

## 7. Census grid

*Needed only for the census-grid method.*

**In Germany:** [download the Zensus 2022 grid (children by age, about 38 MB)](https://www.destatis.de/static/DE/zensus/gitterdaten/Alter_in_infrastrukturellen_Altersgruppen.zip). It is one file for all of Germany, so use it for any German city. Save it on this computer, do not unzip it, and choose it below. Then choose the columns `Unter3` and `a3bis5`. The notebook keeps only the cells around your city.

The data refer to 15 May 2022. Source to credit: Statistische Ämter des Bundes und der Länder, Zensus 2022, Datenlizenz Deutschland - Namensnennung 2.0. More files are on the [Zensus grid page](https://www.zensus2022.de/DE/Ergebnisse-des-Zensus/gitterzellen.html).

**No grid, or outside Germany?** Go back to section 5 and choose *district totals and buildings*.

In [ ]:
steps.census(setup);

## 8. Child locations

*Needed only if you chose "I already have child locations".*

In [ ]:
steps.direct(setup);

## 9. Where new facilities can go

*Needed: a decision.*

In [ ]:
steps.sites(setup);

## 10. Buildings

*Needed for most choices: a map file of building footprints.*

In [ ]:
steps.buildings(setup);

## 11. Priority group

*Needed: one table.* After saving an index you see which districts it selects, so check that they make sense. If a district sits just below the cut-off, say so when you report results.

In [ ]:
steps.priority(setup);

## 12. Review

Check your choices, then click **Create the settings**. If something is listed as still to do, go back to that section.

In [ ]:
steps.review(setup);

## 13. Check the data and build the model inputs

Now the work starts, and the sections below wait for their results. You can use Run All from here if the licence is ready. Keep the computer awake.

This step turns your data into demand locations, walking times and candidate sites. It needs neither Gurobi nor internet. It takes about a minute for the demo city, and about 2 hours for a city of Dortmund's size.

In [ ]:
city = setup.city()
guide.check_data(city)
guide.build_inputs(city)

## 14. Your Gurobi licence

The models are solved with [Gurobi](https://www.gurobi.com/), which needs a licence. Academic licences are free. The next cell explains how your licence is handled before it asks for anything.

In [ ]:
licence = guide.ask_licence(LICENCE, LICENCE_FILE)
guide.check_licence(city, licence)

## 15. Solve the three models

First choose **P**, the number of new facilities, in the box below (default 3), then run the solve cell. The solve cell reads the box when it runs.

Every message appears below as it happens, and a *still working* line appears when the solver is busy. Stopping at a time limit is normal, and the results say which solutions are proven optimal.

**Memory:** a city the size of Münster needs about 10 GB; a larger city needs more, so close other programs first.

In [ ]:
p_box = steps.facility_box()

In [ ]:
run = guide.solve(city, licence, guide.choose_p(p_box.value))

## 16. Results

In [ ]:
results = guide.summarise(city, run)
guide.show_results(results)

**How to read these results**

- Differences between the three models here come from their objectives, because they share the same inputs.
- Only *proven optimal* results are optimal. A *time-limited* result is a feasible solution, not a proven optimum.
- Sites may differ between runs where several solutions are equally good, or where a stage is time-limited.
- Results describe potential walking access. Kita capacity, opening hours and parental choice are not modelled, and candidate sites are not checked for availability.
- Results built with different data levels or methods are not directly comparable.